# Phase 4 — Zero-Backend Interactive Web Dashboard Export (`web/index.html`)

This notebook documents **Phase 4**, which turns our entire SQLite database (`data/galaxy_atlas.db`), downloaded SDSS RGB & FITS Lupton images, and trained ML/CNN evaluation benchmarks into a standalone, zero-backend interactive HTML/CSS/JS website powered by **Plotly.js**.

## How Phase 4 Works
1. **`src/export_dashboard.py`** queries `v_galaxy_dashboard`, `images`, `predictions`, and `metadata` from SQLite plus the saved metrics JSON files from both classifier folders (`classifiers/astronn_and_astroml/` and `classifiers/tensorflow_cnn/`).
2. It writes two compact data bundles:
   - `web/data/atlas_data.json` — standard JSON payload.
   - `web/data/atlas_data.js` — `window.ATLAS_DATA = ...` so you can double-click **`web/index.html`** directly from Windows Explorer with **zero CORS errors** and zero backend server required!
3. It copies downloaded RGB cutouts and FITS Lupton RGB composites into `web/assets/` so the `web/` directory is completely self-contained and ready for GitHub Pages or local viewing.

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ProjectConfig
from src.export_dashboard import export_atlas_dashboard

RUN_EXPORT = True
config = ProjectConfig()

if RUN_EXPORT:
    summary = export_atlas_dashboard(config=config, copy_images=True)
    print(json.dumps(summary, indent=2))

{
  "json_path": "C:\\Things\\Astrophysics_Antigravity_Pj\\AstrophysicsProjectRepo\\web\\data\\atlas_data.json",
  "js_path": "C:\\Things\\Astrophysics_Antigravity_Pj\\AstrophysicsProjectRepo\\web\\data\\atlas_data.js",
  "size_kb": 962.1,
  "galaxies": 2489,
  "rgb_count": 284,
  "showcase_count": 6,
  "models": [
    "rf",
    "astronn",
    "tf"
  ]
}


---
## Inspect the Exported JSON Structure

Notice that every SDSS `objID` is serialized as a string (`"1237645879551066262"`) so JavaScript's 53-bit `Number` type never truncates 64-bit SDSS identifiers.

In [2]:
atlas_json_path = PROJECT_ROOT / "web" / "data" / "atlas_data.json"
payload = json.loads(atlas_json_path.read_text(encoding="utf-8"))

print("Top-level keys:", list(payload.keys()))
print("Summary:", json.dumps({k: v for k, v in payload["summary"].items() if k != "correlation_matrix"}, indent=2))
print("\nSample galaxy record:")
imaged_sample = next((g for g in payload["galaxies"] if g["img"] and g["pred"]), payload["galaxies"][0])
print(json.dumps(imaged_sample, indent=2))

Top-level keys: ['generated_at', 'project', 'summary', 'models', 'showcase', 'galaxies']
Summary: {
  "total_galaxies": 2489,
  "early_count": 1368,
  "late_count": 1121,
  "rgb_cutouts_count": 284,
  "fits_showcase_count": 6,
  "zoo_matched_count": 1908,
  "color_population_crosstab": {
    "early": {
      "red_sequence": 1209,
      "blue_cloud": 159
    },
    "late": {
      "red_sequence": 340,
      "blue_cloud": 781
    }
  },
  "mass_size_fits": {
    "early": {
      "n": 1368,
      "alpha_slope": 0.310728023755794,
      "beta_intercept_at_10p5": 0.43173840716474404,
      "re_kpc_at_10p5": 2.702330152886339,
      "r_squared": 0.4956890223680764,
      "p_value": 2.6867686927960364e-205,
      "stderr": 0.008480073835065213
    },
    "late": {
      "n": 1121,
      "alpha_slope": 0.18499382700011927,
      "beta_intercept_at_10p5": 0.6396446777322546,
      "re_kpc_at_10p5": 4.361588386117929,
      "r_squared": 0.30152905189043366,
      "p_value": 2.7407580428604828e-8

---
## Opening the Interactive Dashboard

- **Instant option (Zero Backend)**: Open `web/index.html` directly in Chrome, Edge, or Firefox.
- **Local HTTP server option**:
  ```powershell
  .\.venv\Scripts\python.exe -m http.server 8000 --directory web
  ```
  Then open `http://localhost:8000` in your browser.